# 04 Feature Engineering

Encodes categorical variables, normalizes numeric columns, adds simple derived features when matching productivity-domain signals are present, and saves to `ai/preprocessing/features`.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'ai' / 'datasets'
PREPROCESSING_DIR = PROJECT_ROOT / 'ai' / 'preprocessing'
MODEL_DIR = PROJECT_ROOT / 'ai' / 'models'

CSV_TSV_EXTENSIONS = {'.csv', '.tsv'}

def discover_data_files(data_dir=DATA_DIR):
    return sorted(p for p in data_dir.rglob('*') if p.suffix.lower() in CSV_TSV_EXTENSIONS)

def read_dataset(path):
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep)

def safe_name(path):
    return path.relative_to(DATA_DIR).as_posix().replace('/', '__')

def display_heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(f'### {text}'))
    except Exception:
        print(f'\n### {text}')

from sklearn.preprocessing import StandardScaler
FEATURE_DIR = PREPROCESSING_DIR / 'features'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

def add_derived_features(df):
    out = df.copy()
    cols = {c.lower(): c for c in out.columns}
    def find(keys):
        for key in keys:
            for low, original in cols.items():
                if key in low and pd.api.types.is_numeric_dtype(out[original]):
                    return original
        return None
    sleep = find(['sleep'])
    stress = find(['stress'])
    workload = find(['workload', 'work_hours', 'hours_worked'])
    if sleep and stress:
        out['sleep_stress_balance'] = out[sleep] / (out[stress].abs() + 1)
    if workload and sleep:
        out['workload_sleep_ratio'] = out[workload] / (out[sleep].abs() + 1)
    numeric = out.select_dtypes(include='number')
    if numeric.shape[1] >= 2:
        out['numeric_mean_signal'] = numeric.mean(axis=1)
    return out

def engineer_features(df):
    enriched = add_derived_features(df)
    numeric_cols = enriched.select_dtypes(include='number').columns.tolist()
    categorical_cols = enriched.select_dtypes(exclude='number').columns.tolist()
    processed = pd.get_dummies(enriched, columns=categorical_cols, dummy_na=False, drop_first=False)
    final_numeric = processed.select_dtypes(include='number').columns
    if len(final_numeric):
        scaler = StandardScaler()
        processed[final_numeric] = scaler.fit_transform(processed[final_numeric].fillna(processed[final_numeric].median()))
    return processed


In [ ]:
for path in discover_data_files():
    display_heading(path.relative_to(DATA_DIR).as_posix())
    try:
        df = read_dataset(path)
        processed = engineer_features(df)
        output = FEATURE_DIR / f'{safe_name(path).rsplit(".", 1)[0]}_features.csv'
        processed.to_csv(output, index=False)
        print('Input shape:', df.shape)
        print('Processed shape:', processed.shape)
        print('Saved:', output.relative_to(PROJECT_ROOT))
    except Exception as exc:
        print(f'Feature engineering failed for {path.name}: {exc}')
